# Notebook 4: Minutes Projection Model

**Goal:** Predict minutes played — the highest-leverage variable for FP accuracy.

**Why this matters:** A player projected for 32 min who plays 18 (foul trouble, blowout) will massively underperform. Minutes errors cascade into FP errors.

**Input:** `yakos_features.parquet`

**Output:** `models/yakos_minutes_model.pkl`

## Evaluation Targets
- MAE ≤ 3 minutes
- Flag players with high minutes variance (>5 min std dev) as 'volatile'

In [ ]:
# !pip install scikit-learn lightgbm joblib pyarrow -q

import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

INPUT_PATH = 'yakos_features.parquet'
MODEL_DIR  = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])

# Filter to rows where minutes is available
df = df.dropna(subset=['minutes'])
print(f'Rows with minutes: {len(df)}')

## Features for Minutes Model

In [ ]:
CUTOFF_DATE = pd.Timestamp('2026-02-15')

MIN_FEATURE_COLS = [
    'salary',
    'rolling_min_5', 'rolling_min_10', 'rolling_min_20', 'min_std_10',
    'vegas_total', 'vegas_spread',
    'b2b', 'days_rest',
    'rolling_usage_10',
]
TARGET = 'minutes'

available_feats = [c for c in MIN_FEATURE_COLS if c in df.columns]
print(f'Using {len(available_feats)} features: {available_feats}')

train_df = df[df['game_date'] < CUTOFF_DATE].dropna(subset=[TARGET])
val_df   = df[df['game_date'] >= CUTOFF_DATE].dropna(subset=[TARGET])

X_train = train_df[available_feats]
y_train = train_df[TARGET]
X_val   = val_df[available_feats]
y_val   = val_df[TARGET]

print(f'Train: {len(X_train)} rows | Val: {len(X_val)} rows')

## Ridge Regression Baseline

In [ ]:
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   Ridge(alpha=5.0)),
])

ridge_pipeline.fit(X_train, y_train)
ridge_preds = ridge_pipeline.predict(X_val)

mae_ridge  = mean_absolute_error(y_val, ridge_preds)
rmse_ridge = np.sqrt(mean_squared_error(y_val, ridge_preds))
bias_ridge = (ridge_preds - y_val).mean()

print('=== Ridge Regression ===')
print(f'  MAE:  {mae_ridge:.2f} min  (target ≤ 3)')
print(f'  RMSE: {rmse_ridge:.2f} min')
print(f'  Bias: {bias_ridge:+.2f} min')

## LightGBM Minutes Model

In [ ]:
try:
    import lightgbm as lgb

    imp = SimpleImputer(strategy='median')
    X_train_imp = imp.fit_transform(X_train)
    X_val_imp   = imp.transform(X_val)

    lgb_model = lgb.LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=15,
        subsample=0.8,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(
        X_train_imp, y_train,
        eval_set=[(X_val_imp, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(100)],
    )
    lgb_preds = lgb_model.predict(X_val_imp)

    mae_lgb  = mean_absolute_error(y_val, lgb_preds)
    rmse_lgb = np.sqrt(mean_squared_error(y_val, lgb_preds))
    bias_lgb = (lgb_preds - y_val).mean()

    print('=== LightGBM ===')
    print(f'  MAE:  {mae_lgb:.2f} min  (target ≤ 3)')
    print(f'  RMSE: {rmse_lgb:.2f} min')
    print(f'  Bias: {bias_lgb:+.2f} min')

    BEST_MODEL = 'lgb'
except ImportError:
    print('LightGBM not available. Using Ridge.')
    BEST_MODEL = 'ridge'

## Volatile Players (High Minutes Variance)

In [ ]:
if 'min_std_10' in df.columns:
    volatile_threshold = 5.0  # minutes std dev
    volatile = (
        df.groupby('player_name')['min_std_10']
        .mean()
        .rename('avg_min_std')
        .reset_index()
        .query('avg_min_std >= @volatile_threshold')
        .sort_values('avg_min_std', ascending=False)
    )
    print(f'Volatile players (min std ≥ {volatile_threshold}): {len(volatile)}')
    print(volatile.head(10).to_string())

## Save Minutes Model

In [ ]:
model_to_save = lgb_model if BEST_MODEL == 'lgb' else ridge_pipeline
model_path = MODEL_DIR / 'yakos_minutes_model.pkl'
joblib.dump(model_to_save, model_path)
print(f'Minutes model saved: {model_path}')

if BEST_MODEL == 'lgb':
    joblib.dump(imp, MODEL_DIR / 'yakos_minutes_imputer.pkl')

with open(MODEL_DIR / 'yakos_minutes_features.json', 'w') as f:
    json.dump(available_feats, f)

metrics = {
    'model_type': BEST_MODEL,
    'mae': float(mae_lgb if BEST_MODEL == 'lgb' else mae_ridge),
    'bias': float(bias_lgb if BEST_MODEL == 'lgb' else bias_ridge),
}
with open(MODEL_DIR / 'yakos_minutes_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Minutes model metrics:', metrics)